# NACC_PrimeKGL
Copied from the original project. Paths are relative to the main folder.
Full PrimeKG embeddings and entity indices are shared by all four cohorts.


In [ ]:
from pathlib import Path
import os
import sys
_starts = [Path.cwd(), Path(globals().get("__file__", Path.cwd() / "notebook")).resolve().parent]
CLINICALKG_ROOT = next(p for start in _starts for p in (start, *start.parents)
                      if (p / "run" / "common" / "paths.py").is_file())
sys.path.insert(0, str(CLINICALKG_ROOT.parent))
from clinicalKG.run.common.paths import PROJECT_ROOT, SHARED_PRIMEKG, ANALYSIS_ROOT, raw_primekg_path, work_dir, asset_path, dataset_dir, result_path
DATASET = 'NACC'
METHOD = 'PrimeKGL'
WORK_DIR = work_dir(DATASET, METHOD) if METHOD != "CustomKG" else CLINICALKG_ROOT / "run" / "common"
DATASET_DIR = dataset_dir(DATASET)
os.environ.setdefault("KG_ABLATION_OUTPUT", str(ANALYSIS_ROOT))
if DATASET_DIR.is_dir():
    os.chdir(DATASET_DIR)


In [ ]:
# Full PrimeKG is used directly: no seed extraction, blacklist, or Top-k.
RAW_KG_PATH = str(raw_primekg_path())
SHARED_PRIMEKG.mkdir(parents=True, exist_ok=True)
if not Path(RAW_KG_PATH).is_file():
    raise FileNotFoundError(RAW_KG_PATH)
print("Full PrimeKG source:", RAW_KG_PATH)
print("All cohorts share graph-trained assets in:", SHARED_PRIMEKG)


In [ ]:
from pathlib import Path
import sys
_split_root = CLINICALKG_ROOT
if str(_split_root) not in sys.path:
    sys.path.insert(0, str(_split_root))
from clinicalKG.run.common.experiment_split import split_cohort_arrays, cohort_indices
# Fixed patient split: about 64% train, 16% validation, 20% test.
# Keep this split unchanged across model variants and all five training runs.

import pandas as pd
import numpy as np
import json
import warnings

warnings.filterwarnings('ignore')

NACC_DATA_PATH = str(DATASET_DIR / 'NACC诊断.csv')
SEED_CSV_PATH = str(WORK_DIR / 'NACC_Seed_DementiaHKG.csv')
REFINED_KG_PATH = str(raw_primekg_path())

OUTPUT_GRAPH_PATH = str(SHARED_PRIMEKG / 'PrimeKG-G.csv')
OUTPUT_DICT_PATH = str(WORK_DIR / 'NACC-PrimeKG-Entities.json')

OUTPUT_SPLIT_PATH = str(WORK_DIR / 'NACC_Dataset_Split.csv')

MAX_SEQ_LEN = 45
PAD_TOKEN = "[PAD]"

lower_is_worse = ['boston', 'mint', 'animal', 'Fwords', 'lm_imm', 'craft_imm', 'lm_del', 'craft_del', 'digitB', 'digitBL', 'digitF', 'digitFL', 'numberB', 'numberBL', 'numberF', 'numberFL']
higher_is_worse = ['trailA', 'trailB']
MIN_SAMPLE_SIZE = 30

def build_patient_mapping_nacc():
    print("==================================================")
    print("🔗 阶段 3: 构建患者动态映射与客观图谱格式化 (包含安全的数据集划分与常模机制)...")

    try:
        kg_df = pd.read_csv(REFINED_KG_PATH)
        triplets_df = kg_df[['x_name', 'relation', 'y_name']].copy()
        triplets_df.columns = ['head', 'relation', 'tail']
        
        triplets_df.to_csv(OUTPUT_GRAPH_PATH, index=False, encoding='utf-8')
        print(f"   ✅ 标准化图谱已生成: {OUTPUT_GRAPH_PATH} (共 {len(triplets_df)} 条边)")
        print("   🔒 [安全检查]: 该图谱中不存在任何患者节点，已彻底阻断 Transductive 泄露。")
    except FileNotFoundError:
        print(f"   ❌ 找不到 {REFINED_KG_PATH}，请确保上一阶段已成功运行。")
        return

    seed_df = pd.read_csv(SEED_CSV_PATH)
    mapping_df = seed_df[seed_df['Category'] == 'Clinical_Mapping']
    nacc_to_pkg_map = dict(zip(mapping_df['NACC_Feature'], mapping_df['PrimeKG_Node']))
    print(f"\n   📊 成功加载 {len(nacc_to_pkg_map)} 条临床特征映射规则。")

    try:
        nacc_df = pd.read_csv(NACC_DATA_PATH, low_memory=False)
    except FileNotFoundError:
        print(f"   ❌ 找不到患者数据 {NACC_DATA_PATH}。")
        return

    print("\n   🔀 正在执行安全的随机数据集划分 (Train: 70%, Val: 10%, Test: 20%)...")
    np.random.seed(42) 
    
    def get_pid(row):
        if 'filename' in row and pd.notna(row['filename']): return str(row['filename']).strip()
        if 'ID' in row and pd.notna(row['ID']): return str(row['ID']).strip()
        return None

    nacc_df['patient_id_temp'] = nacc_df.apply(get_pid, axis=1)
    # Reuse the fixed patient split used by every downstream model.
    from clinicalKG.run.common.experiment_split import export_manifest, training_rows
    export_manifest("NACC", OUTPUT_SPLIT_PATH)

    train_df = nacc_df.loc[training_rows("NACC", nacc_df)].copy()

    train_df['age'] = pd.to_numeric(train_df['age'], errors='coerce')
    train_df['education'] = pd.to_numeric(train_df['education'], errors='coerce')
    train_df['age_group'] = pd.cut(train_df['age'], bins=[0, 65, 75, 120], labels=['<65', '65-75', '>75'])
    train_df['edu_group'] = pd.cut(train_df['education'], bins=[0, 12, 30], labels=['<=12', '>12'])

    nacc_df['age'] = pd.to_numeric(nacc_df['age'], errors='coerce')
    nacc_df['education'] = pd.to_numeric(nacc_df['education'], errors='coerce')
    nacc_df['age_group'] = pd.cut(nacc_df['age'], bins=[0, 65, 75, 120], labels=['<65', '65-75', '>75'])
    nacc_df['edu_group'] = pd.cut(nacc_df['education'], bins=[0, 12, 30], labels=['<=12', '>12'])

    print("   🧠 正在基于训练集 (NC组) 计算人口学常模...")
    nc_df = train_df[train_df['NC'] == 1].copy()
    normative_stats = {}
    
    global_nc_stats = {}
    for col in lower_is_worse + higher_is_worse:
        if col in nc_df.columns:
            nc_df[col] = pd.to_numeric(nc_df[col], errors='coerce')
            valid_nc = nc_df[~nc_df[col].isin([88, 99, 88.0, 99.0, -4, -1])]
            global_nc_stats[col] = {'mean': valid_nc[col].mean(), 'std': valid_nc[col].std()}

    for col in lower_is_worse + higher_is_worse:
        if col in nc_df.columns:
            valid_nc = nc_df[~nc_df[col].isin([88, 99, 88.0, 99.0, -4, -1])]
            stats = valid_nc.groupby(['age_group', 'edu_group'], observed=False)[col].agg(['mean', 'std', 'count']).to_dict(orient='index')
            
            safe_stats = {}
            for group_key, stat_values in stats.items():
                if pd.notna(stat_values['count']) and stat_values['count'] >= MIN_SAMPLE_SIZE:
                    safe_stats[group_key] = stat_values
                else:
                    safe_stats[group_key] = global_nc_stats.get(col, {'mean': np.nan, 'std': np.nan})
            normative_stats[col] = safe_stats

    def check_abnormal(col_name, val, row_data):
        if col_name not in lower_is_worse and col_name not in higher_is_worse:
            if col_name == 'gds': return val >= 5
            return val > 0
            
        age_grp = row_data.get('age_group')
        edu_grp = row_data.get('edu_group')
        
        if col_name in normative_stats and pd.notna(age_grp) and pd.notna(edu_grp):
            group_stat = normative_stats[col_name].get((age_grp, edu_grp))
            if group_stat and pd.notna(group_stat['mean']) and pd.notna(group_stat['std']):
                mean = group_stat['mean']
                std = max(group_stat['std'], 1e-5) 
                z_score = (val - mean) / std
                
                if col_name in lower_is_worse: return z_score < -1.5
                else: return z_score > 1.5
                    
        if col_name in ['boston', 'mint', 'animal', 'Fwords']: return val < 15
        elif col_name in ['lm_imm', 'craft_imm', 'lm_del', 'craft_del']: return val <= 8
        elif col_name == 'trailA': return val > 78
        elif col_name == 'trailB': return val > 273
        elif col_name in ['digitB', 'digitBL', 'digitF', 'digitFL', 'numberB', 'numberBL', 'numberF', 'numberFL']: return val < 5
        return False

    patient_entities = {}
    valid_mapped_patients = 0
    total_features_mapped = 0

    print(f"\n   ⏳ 正在为所有患者建立图谱实体序列 (统一输出以保持下游文件不变)...")
    
    for index, row in nacc_df.iterrows():
        patient_id = row.get('patient_id_temp')
        if not patient_id: continue

        active_nodes = []
        for col_name, val in row.items():
            if col_name in nacc_to_pkg_map:
                val_str = str(val).strip().lower()
                if pd.notna(val) and val_str not in ['', 'nan', '-4', '-1', '88', '99', '88.0', '99.0']:
                    try:
                        num_val = float(val)
                        if check_abnormal(col_name, num_val, row):
                            primekg_node = nacc_to_pkg_map[col_name]
                            active_nodes.append(primekg_node)
                            total_features_mapped += 1
                    except ValueError:
                        pass

        active_nodes = list(set(active_nodes))
        seq_len = len(active_nodes)
        
        if seq_len > MAX_SEQ_LEN:
            active_nodes = active_nodes[:MAX_SEQ_LEN]
            attention_mask = [1] * MAX_SEQ_LEN
        else:
            pad_length = MAX_SEQ_LEN - seq_len
            attention_mask = [1] * seq_len + [0] * pad_length
            active_nodes = active_nodes + [PAD_TOKEN] * pad_length

        patient_entities[patient_id] = {
            "input_nodes": active_nodes,
            "attention_mask": attention_mask
        }
        
        if seq_len > 0:
            valid_mapped_patients += 1

    with open(OUTPUT_DICT_PATH, 'w', encoding='utf-8') as f:
        json.dump(patient_entities, f, indent=4, ensure_ascii=False)

    print("\n==================================================")
    print("💾 阶段 3 任务完成！")
    print(f"   👨‍⚕️ 处理患者总数: {len(nacc_df)}")
    print(f"   🔗 成功映射患者数: {valid_mapped_patients}")
    print(f"   📈 映射特征总次: {total_features_mapped}")
    print(f"   📄 图谱序列已保存至: {OUTPUT_DICT_PATH} (文件格式与名称均未改变)")
    print("==================================================")

if __name__ == "__main__":
    build_patient_mapping_nacc()

In [ ]:
# Same DistMult score/loss; one shared full-graph pretraining job.
from clinicalKG.PrimeKGL.full_primekg import train_full_primekg, plot_loss
RESUME = False  # Set True to continue PrimeKG-DistMult-last.pt.
NUM_EPOCHS = 50  # Total target epochs, shared across cohorts.
EMBED_DIM = 128
BATCH_SIZE = 512
LR = 0.005
training_state = train_full_primekg(epochs=NUM_EPOCHS, embed_dim=EMBED_DIM,
                                   batch_size=BATCH_SIZE, lr=LR, resume=RESUME)
plot_loss()
